# Eddies in the Bellingshausen Sea, Antarctica

Author: Suyue Li 

Note: ChatGPT has been used to generate some of the codes for this project 

In [ ]:
import numpy as np
import os
import glob
import re
import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from pathlib import Path
from pprint import pprint
import pandas as pd
import requests
import earthaccess

import swot_ssh_utils as swot

Test the number of netcdf files in the study region.

In [2]:
# mt_stream_bbox = (-69.5, -68.5, -67.5, -67)
mt_stream_bbox = (-69.5, -67.5, -68.5, -67)

swot_results = earthaccess.search_data(
    short_name="SWOT_L2_LR_SSH_D",
    cloud_hosted=True,
    temporal=("2023-05-18", "2023-05-22"),
    bounding_box=mt_stream_bbox,
)


unsmoothed = [g for g in swot_results if "Unsmoothed" in repr(g)]
len(unsmoothed)

2

Download these netcdf files...

In [11]:
extent = [-69.5, -68.5, -67.5, -67]  # [lon_min, lon_max, lat_min, lat_max]
lon_min, lon_max, lat_min, lat_max = extent
mt_stream_bbox = (lon_min, lat_min, lon_max, lat_max)  # earthaccess: (W, S, E, N)

download_dir = "./swot_data"
os.makedirs(download_dir, exist_ok=True)

# =============================================================
# 1. Auth & Search
# =============================================================
auth = earthaccess.login()

swot_results = earthaccess.search_data(
    short_name="SWOT_L2_LR_SSH_D",
    cloud_hosted=True,
    temporal=("2023-01-01", "2023-07-10"),
    bounding_box=mt_stream_bbox,
)

unsmoothed = [g for g in swot_results if "Unsmoothed" in repr(g)]
print(f"Found {len(unsmoothed)} unsmoothed granules from search")

# =============================================================
# 2. Download
# =============================================================
print("Downloading granules...")
local_files = earthaccess.download(unsmoothed, download_dir)
print(f"Downloaded {len(local_files)} files to {download_dir}/")

Found 102 unsmoothed granules from search


QUEUEING TASKS | :   0%|          | 0/102 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/102 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/102 [00:00<?, ?it/s]

Downloaded 102 files to ./swot_data/


In [4]:
def extract_cycle_number(filename):
    """
    Try to extract cycle number from SWOT filename.
    Adjust this if your filename format differs.
    """
    basename = os.path.basename(filename)
    parts = basename.split('_')

    for p in parts:
        if p.isdigit() and len(p) >= 3:
            val = int(p)
            if 0 <= val <= 9999:
                return val

    m = re.search(r'cycle[_\-]?(\d+)', basename, re.IGNORECASE)
    if m:
        return int(m.group(1))

    return None

# This is simple bin-average gridding, not a sophisticated interpolation method.
def make_gridded_field(lon, lat, val, lon_min, lon_max, lat_min, lat_max, nx=400, ny=400):
    """
    Bin swath data onto a regular grid for imshow plotting.
    """
    mask = np.isfinite(lon) & np.isfinite(lat) & np.isfinite(val)
    mask &= (lon >= lon_min) & (lon <= lon_max) & (lat >= lat_min) & (lat <= lat_max)

    if not np.any(mask):
        return None

    lon1 = lon[mask].ravel()
    lat1 = lat[mask].ravel()
    val1 = val[mask].ravel()

    lon_edges = np.linspace(lon_min, lon_max, nx + 1)
    lat_edges = np.linspace(lat_min, lat_max, ny + 1)

    sum_grid, _, _ = np.histogram2d(lat1, lon1, bins=[lat_edges, lon_edges], weights=val1)
    cnt_grid, _, _ = np.histogram2d(lat1, lon1, bins=[lat_edges, lon_edges])

    with np.errstate(invalid='ignore', divide='ignore'):
        grid = sum_grid / cnt_grid

    grid[cnt_grid == 0] = np.nan
    return grid

## Plot SSHA and sigma0 figures 

In [5]:
extent = [-69, -66, -67.5, -66.5]
lon_min, lon_max, lat_min, lat_max = extent

data_dir = "./swot_data"
out_dir = "./calvalplot"
os.makedirs(out_dir, exist_ok=True)

cycle_min = 473
cycle_max = 577

grid_nx = 400
grid_ny = 400

all_nc = sorted(glob.glob(os.path.join(data_dir, "*.nc")))
all_nc = [f for f in all_nc if "Unsmoothed" in os.path.basename(f)]
all_nc = [f for f in all_nc if "PIC2" not in os.path.basename(f)]

filtered_nc = []
for f in all_nc:
    cyc = extract_cycle_number(f)
    if cyc is not None and cycle_min <= cyc <= cycle_max:
        filtered_nc.append(f)

all_nc = filtered_nc

print(f"Found {len(all_nc)} unsmoothed files (excluding PIC2) with cycle between {cycle_min} and {cycle_max}")

# =============================================================
# Plot helpers
# =============================================================
land = cfeature.NaturalEarthFeature(
    "physical", "land", "10m",
    edgecolor="black", facecolor="#e0e0e0", linewidth=0.5
)

bad_threshold = 2**31
saved_count = 0

# =============================================================
# Main loop
# =============================================================
for i, fn in enumerate(all_nc):
    basename = os.path.basename(fn)
    print(f"[{i+1}/{len(all_nc)}] {basename}")

    try:
        dd = swot.SSH_L2()
        dd.load_data(fn, lat_bounds=[-75, -60])
    except Exception as e:
        print(f"  SKIP (load failed): {e}")
        continue

    parts = basename.split('_')
    try:
        title_id = f"{parts[5]}_{parts[6]}_{parts[7]}"
    except Exception:
        title_id = basename.replace('.nc', '')

    fig, (ax1, ax2) = plt.subplots(
        1, 2,
        figsize=(26, 9),
        subplot_kw={'projection': ccrs.PlateCarree()}
    )

    has_ssha = False
    has_sig0 = False

    ssha_lon_all = []
    ssha_lat_all = []
    ssha_val_all = []

    sig0_lon_all = []
    sig0_lat_all = []
    sig0_val_all = []

    for side_label, tmp in [("left", dd.left), ("right", dd.right)]:
        if tmp is None:
            continue

        try:
            lon_raw = tmp.longitude.values
            lat_raw = tmp.latitude.values
            surf_flag = tmp.ancillary_surface_classification_flag.values
            surf_ocean = (surf_flag == 0)
        except Exception:
            continue

        lon = np.where(lon_raw > 180, lon_raw - 360, lon_raw)

        spatial_mask = (
            (lon >= lon_min) & (lon <= lon_max) &
            (lat_raw >= lat_min) & (lat_raw <= lat_max)
        )

        # ---------------------------------------------------------
        # Panel 1: SSHA
        # ---------------------------------------------------------
        try:
            ssha = tmp.ssha_karin_2.values.astype(float)
            xover = tmp.height_cor_xover.values.astype(float)
        except Exception as e:
            print(f"  {side_label} SSHA read failed: {e}")
            ssha = None

        if ssha is not None:
            if "height_cor_xover_qual" in tmp.variables:
                xover_qual = tmp.height_cor_xover_qual.values
                xover = np.where(xover_qual < bad_threshold, xover, np.nan)

            dtm = ssha + xover
            dtm = np.where(surf_ocean, dtm, np.nan)

            if "ssha_karin_2_qual" in tmp.variables:
                qc = tmp.ssha_karin_2_qual.values
                dtm = np.where(qc < bad_threshold, dtm, np.nan)

            m_ssha = spatial_mask & np.isfinite(dtm)
            print(f"  {side_label} SSHA: {np.sum(m_ssha)} valid points")

            if np.any(m_ssha):
                has_ssha = True
                ssha_lon_all.append(lon[m_ssha])
                ssha_lat_all.append(lat_raw[m_ssha])
                ssha_val_all.append(dtm[m_ssha])

        # ---------------------------------------------------------
        # Panel 2: sig0_karin_2
        # ---------------------------------------------------------
        try:
            sig0 = tmp.sig0_karin_2.values.astype(float)
        except AttributeError:
            print(f"  {side_label}: sig0_karin_2 not found")
            sig0 = None

        if sig0 is not None:
            sig0 = np.where(surf_ocean, sig0, np.nan)

            if "sig0_karin_2_qual" in tmp.variables:
                sig0_qc = tmp.sig0_karin_2_qual.values
                sig0 = np.where(sig0_qc < bad_threshold, sig0, np.nan)

            m_sig0 = spatial_mask & np.isfinite(sig0)
            print(f"  {side_label} sig0: {np.sum(m_sig0)} valid points")

            if np.any(m_sig0):
                has_sig0 = True
                sig0_lon_all.append(lon[m_sig0])
                sig0_lat_all.append(lat_raw[m_sig0])
                sig0_val_all.append(sig0[m_sig0])

    if not has_ssha and not has_sig0:
        print("  SKIP (no valid data)")
        plt.close(fig)
        continue

    # ============================================================
    # Render SSHA 
    # ============================================================
    if has_ssha:
        lon_cat = np.concatenate(ssha_lon_all)
        lat_cat = np.concatenate(ssha_lat_all)
        val_cat = np.concatenate(ssha_val_all)

        ssha_grid = make_gridded_field(
            lon_cat, lat_cat, val_cat,
            lon_min, lon_max, lat_min, lat_max,
            nx=grid_nx, ny=grid_ny
        )

        if ssha_grid is not None:
            im1 = ax1.imshow(
                ssha_grid,
                extent=[lon_min, lon_max, lat_min, lat_max],
                origin='lower',
                cmap='RdBu_r',
                vmin=-0.30,
                vmax=0.30,
                transform=ccrs.PlateCarree(),
                zorder=2,
                interpolation='nearest',
                aspect='auto'
            )
            cbar1 = plt.colorbar(
                im1, ax=ax1, orientation="vertical",
                pad=0.04, shrink=0.7, extend="both"
            )
            cbar1.set_label(
                "SSHA (ssha_karin_2 + height_cor_xover) [m]",
                fontsize=11, weight="bold"
            )

    # ============================================================
    # Render sigma0 
    # ============================================================
    if has_sig0:
        lon_cat = np.concatenate(sig0_lon_all)
        lat_cat = np.concatenate(sig0_lat_all)
        val_cat = np.concatenate(sig0_val_all)

        sig0_grid = make_gridded_field(
            lon_cat, lat_cat, val_cat,
            lon_min, lon_max, lat_min, lat_max,
            nx=grid_nx, ny=grid_ny
        )

        if sig0_grid is not None:
            im2 = ax2.imshow(
                sig0_grid,
                extent=[lon_min, lon_max, lat_min, lat_max],
                origin='lower',
                cmap='gray',
                vmin=5,
                vmax=25,
                transform=ccrs.PlateCarree(),
                zorder=2,
                interpolation='nearest',
                aspect='auto'
            )
            cbar2 = plt.colorbar(
                im2, ax=ax2, orientation="vertical",
                pad=0.04, shrink=0.7
            )
            cbar2.set_label(
                r"$\sigma_0$ (sig0_karin_2) [dB]",
                fontsize=11, weight="bold"
            )

    # ============================================================
    # Common map formatting
    # ============================================================
    for ax in [ax1, ax2]:
        ax.add_feature(land, zorder=1)
        ax.add_feature(cfeature.COASTLINE, linewidth=0.8, zorder=4)
        ax.set_extent(extent, crs=ccrs.PlateCarree())

        gl = ax.gridlines(
            draw_labels=True, dms=True, x_inline=False, y_inline=False,
            color="gray", alpha=0.3, linestyle="--"
        )
        gl.top_labels = False
        gl.right_labels = False

    ax1.set_title(f"SSHA — {title_id}", fontsize=13, pad=15, fontweight="bold")
    ax2.set_title(f"σ₀ — {title_id}", fontsize=13, pad=15, fontweight="bold")

    fig.suptitle(
        f"SWOT Pass: {title_id} (left + right)",
        fontsize=15, fontweight="bold", y=1.02
    )

    plt.tight_layout()

    save_path = os.path.join(out_dir, f"ssha_sig0_{title_id}.png")
    fig.savefig(save_path, dpi=200, bbox_inches="tight")
    plt.close(fig)

    saved_count += 1
    print(f"  Saved: {save_path}")

print(f"\n{'='*50}")
print(f"Done. {saved_count} figures saved to {out_dir}/")

Found 92 unsmoothed files (excluding PIC2) with cycle between 473 and 577
[1/92] SWOT_L2_LR_SSH_Unsmoothed_483_024_20230407T161255_20230407T170401_PGD0_01.nc
Load unsmoothed data to self.left and self.right
Subset data between latitude bounds  [-75, -60]
  left SSHA: 47206 valid points
  left sig0: 47206 valid points
  right SSHA: 53896 valid points
  right sig0: 53896 valid points
  Saved: ./calvalplot2/ssha_sig0_483_024_20230407T161255.png
[2/92] SWOT_L2_LR_SSH_Unsmoothed_484_024_20230408T160334_20230408T165439_PGD0_01.nc
Load unsmoothed data to self.left and self.right
Subset data between latitude bounds  [-75, -60]
  left SSHA: 47210 valid points
  left sig0: 47210 valid points
  right SSHA: 53895 valid points
  right sig0: 53895 valid points
  Saved: ./calvalplot2/ssha_sig0_484_024_20230408T160334.png
[3/92] SWOT_L2_LR_SSH_Unsmoothed_485_024_20230409T155412_20230409T164517_PGD0_01.nc
Load unsmoothed data to self.left and self.right
Subset data between latitude bounds  [-75, -60]


/home/jovyan/tmp2/ocng-489-689-2026-spring/swot_ssh_utils.py:453: RuntimeWarning: Mean of empty slice
  lat=np.nanmean(data_in['latitude'].data,axis=-1)
/home/jovyan/tmp2/ocng-489-689-2026-spring/swot_ssh_utils.py:453: RuntimeWarning: Mean of empty slice
  lat=np.nanmean(data_in['latitude'].data,axis=-1)


  left SSHA: 47216 valid points
  left sig0: 47216 valid points
  right SSHA: 53866 valid points
  right sig0: 53866 valid points
  Saved: ./calvalplot2/ssha_sig0_485_024_20230409T155412.png
[4/92] SWOT_L2_LR_SSH_Unsmoothed_486_024_20230410T154450_20230410T163555_PGD0_01.nc
Load unsmoothed data to self.left and self.right
Subset data between latitude bounds  [-75, -60]
  left SSHA: 47213 valid points
  left sig0: 47213 valid points
  right SSHA: 53829 valid points
  right sig0: 53829 valid points
  Saved: ./calvalplot2/ssha_sig0_486_024_20230410T154450.png
[5/92] SWOT_L2_LR_SSH_Unsmoothed_487_024_20230411T153528_20230411T162633_PGD0_01.nc
Load unsmoothed data to self.left and self.right
Subset data between latitude bounds  [-75, -60]
  left SSHA: 47242 valid points
  left sig0: 47242 valid points
  right SSHA: 53806 valid points
  right sig0: 53806 valid points
  Saved: ./calvalplot2/ssha_sig0_487_024_20230411T153528.png
[6/92] SWOT_L2_LR_SSH_Unsmoothed_488_024_20230412T152606_20230412

KeyboardInterrupt: 

Error in callback <function flush_figures at 0x7fba69a18400> (for post_execute), with arguments args (),kwargs {}:


KeyboardInterrupt: 

## Plot days against the cropped region ssha

Plot a timeline figure with rotated, tightly-cropped SWOT SSH images placed along
the y-axis according to their acquisition date.

In [15]:
import math
from matplotlib.path import Path
from PIL import Image

import swot_ssh_utils as swot

# ============================================================
# Config
# ============================================================
data_dir = "./swot_data"
out_dir = "./crop_output"
os.makedirs(out_dir, exist_ok=True)

bad_threshold = 2**31
side_to_use = "right"   # "left" or "right"
vmin, vmax = -0.30, 0.30

# latitude band to isolate
lat_min_band, lat_max_band = -67.5, -66.5

# polygon in IMAGE coordinates after latitude-band crop
polygon = np.array([
    [150, 300],
    [240, 360],
    [240, 300],
    [150, 240]
])

# rotation angle for final saved cropped figure
rotate_angle_deg = -40

# cycle range
cycle_min = 400
cycle_max = 600

# file pattern
file_ext = ".nc"

# ============================================================
# Helper functions
# ============================================================
def crop_to_valid(data2d):
    valid = np.isfinite(data2d)
    if not np.any(valid):
        return None, None
    rows = np.where(np.any(valid, axis=1))[0]
    cols = np.where(np.any(valid, axis=0))[0]
    r0, r1 = rows[0], rows[-1]
    c0, c1 = cols[0], cols[-1]
    cropped = data2d[r0:r1+1, c0:c1+1]
    return cropped, (r0, r1, c0, c1)

def compute_ssha(tmp, bad_threshold=2**31):
    lon_raw = tmp.longitude.values
    lat_raw = tmp.latitude.values
    lon = np.where(lon_raw > 180, lon_raw - 360, lon_raw)

    ssha = tmp.ssha_karin_2.values.astype(float)
    xover = tmp.height_cor_xover.values.astype(float)

    if "height_cor_xover_qual" in tmp.variables:
        xover_qual = tmp.height_cor_xover_qual.values
        xover = np.where(xover_qual < bad_threshold, xover, np.nan)

    dtm = ssha + xover

    surf_flag = tmp.ancillary_surface_classification_flag.values
    surf_ocean = (surf_flag == 0)
    dtm = np.where(surf_ocean, dtm, np.nan)

    if "ssha_karin_2_qual" in tmp.variables:
        qc = tmp.ssha_karin_2_qual.values
        dtm = np.where(qc < bad_threshold, dtm, np.nan)

    return lon, lat_raw, dtm

def polygon_mask_from_vertices(shape, polygon_xy):
    nrows, ncols = shape
    xg, yg = np.meshgrid(np.arange(ncols), np.arange(nrows))
    pts = np.vstack((xg.ravel(), yg.ravel())).T
    poly_path = Path(polygon_xy)
    mask = poly_path.contains_points(pts).reshape(nrows, ncols)
    return mask

def rotate_saved_png(infile, outfile, angle_deg, expand=True):
    img = Image.open(infile)
    img_rot = img.rotate(angle_deg, expand=expand, resample=Image.BICUBIC)
    img_rot.save(outfile)

def extract_cycle_number(filename):
    """
    Extract cycle number from SWOT filename:
    SWOT_L2_LR_SSH_Unsmoothed_474_024_...
                              ^^^
    """
    base = os.path.basename(filename)
    m = re.search(r"SWOT_L2_LR_SSH_Unsmoothed_(\d+)_", base)
    if m:
        return int(m.group(1))
    return None

def list_cycle_files(data_dir, cycle_min, cycle_max, file_ext=".nc"):
    files = []
    for fn in os.listdir(data_dir):
        if not fn.endswith(file_ext):
            continue
        cycle = extract_cycle_number(fn)
        if cycle is None:
            continue
        if cycle_min <= cycle <= cycle_max:
            files.append((cycle, os.path.join(data_dir, fn)))
    files.sort(key=lambda x: x[0])
    return files

def process_one_file(fn, out_dir):
    cycle = extract_cycle_number(fn)
    print(f"\nProcessing cycle {cycle}: {os.path.basename(fn)}")

    dd = swot.SSH_L2()
    dd.load_data(fn, lat_bounds=[-75, -60])

    tmp = dd.left if side_to_use == "left" else dd.right
    if tmp is None:
        print(f"  Skipped: {side_to_use} swath missing")
        return None

    try:
        lon, lat_raw, dtm = compute_ssha(tmp, bad_threshold=bad_threshold)
    except Exception as e:
        print(f"  Failed during SSHA computation: {e}")
        return None

    lat_band_mask = (lat_raw >= lat_min_band) & (lat_raw <= lat_max_band)
    dtm_latband = np.where(lat_band_mask, dtm, np.nan)
    dtm_latband_crop, crop_bounds = crop_to_valid(dtm_latband)

    if dtm_latband_crop is None:
        print("  Skipped: no valid data in latitude band")
        return None

    mask_poly = polygon_mask_from_vertices(dtm_latband_crop.shape, polygon)
    dtm_subset = np.where(mask_poly, dtm_latband_crop, np.nan)
    dtm_subset_crop, subset_bounds = crop_to_valid(dtm_subset)

    if dtm_subset_crop is None:
        print("  Skipped: polygon subset contains no valid data")
        return None

    # Save clean cropped figure
    clean_path = os.path.join(out_dir, f"cycle_{cycle:03d}_polygon_subset_{side_to_use}_clean.png")
    fig, ax = plt.subplots(figsize=(3, 4))
    ax.imshow(
        dtm_subset_crop,
        origin="lower",
        cmap="RdBu_r",
        vmin=vmin,
        vmax=vmax,
        interpolation="nearest",
        aspect="auto"
    )
    ax.set_axis_off()
    fig.savefig(clean_path, dpi=200, bbox_inches="tight", pad_inches=0)
    plt.close(fig)

    # Rotate saved image
    rotated_path = os.path.join(out_dir, f"cycle_{cycle:03d}_polygon_subset_{side_to_use}_rotated.png")
    rotate_saved_png(clean_path, rotated_path, angle_deg=rotate_angle_deg, expand=True)

    print("  Saved:", rotated_path)

    return {
        "cycle": cycle,
        "file": fn,
        "data": dtm_subset_crop,
        "clean_path": clean_path,
        "rotated_path": rotated_path
    }

# ============================================================
# Find files in cycle range
# ============================================================
cycle_files = list_cycle_files(data_dir, cycle_min, cycle_max, file_ext=file_ext)

if len(cycle_files) == 0:
    raise ValueError(f"No SWOT files found in {data_dir} for cycles {cycle_min} to {cycle_max}")

print(f"Found {len(cycle_files)} files in cycle range [{cycle_min}, {cycle_max}]")

# ============================================================
# Process all files
# ============================================================
results = []
for cycle, fn in cycle_files:
    out = process_one_file(fn, out_dir)
    if out is not None:
        results.append(out)

if len(results) == 0:
    raise ValueError("No valid cropped subsets were produced.")

print(f"\nGenerated {len(results)} rotated cropped figures.")

# ============================================================
# Plot all cropped subsets in one summary figure
# ============================================================
n = len(results)
ncols = 4
nrows = math.ceil(n / ncols)

fig, axes = plt.subplots(nrows, ncols, figsize=(4*ncols, 4*nrows))
axes = np.atleast_1d(axes).ravel()

for ax, item in zip(axes, results):
    img = Image.open(item["rotated_path"])
    ax.imshow(img)
    ax.set_title(f"Cycle {item['cycle']}")
    ax.set_axis_off()

for ax in axes[len(results):]:
    ax.set_axis_off()

plt.tight_layout()
summary_path = os.path.join(
    out_dir,
    f"all_rotated_polygon_subsets_cycles_{cycle_min}_{cycle_max}_{side_to_use}.png"
)
fig.savefig(summary_path, dpi=200, bbox_inches="tight")
plt.close(fig)

print("Saved summary figure:", summary_path)

Found 102 files in cycle range [400, 600]

Processing cycle 473: SWOT_L2_LR_SSH_Unsmoothed_473_024_20230328T174638_20230328T183743_PGD0_02.nc
Load unsmoothed data to self.left and self.right
Subset data between latitude bounds  [-75, -60]


/home/jovyan/tmp2/ocng-489-689-2026-spring/swot_ssh_utils.py:453: RuntimeWarning: Mean of empty slice
  lat=np.nanmean(data_in['latitude'].data,axis=-1)
/home/jovyan/tmp2/ocng-489-689-2026-spring/swot_ssh_utils.py:453: RuntimeWarning: Mean of empty slice
  lat=np.nanmean(data_in['latitude'].data,axis=-1)


  Skipped: no valid data in latitude band

Processing cycle 474: SWOT_L2_LR_SSH_Unsmoothed_474_024_20230329T173715_20230329T182821_PGD0_01.nc
Load unsmoothed data to self.left and self.right
Subset data between latitude bounds  [-75, -60]


/home/jovyan/tmp2/ocng-489-689-2026-spring/swot_ssh_utils.py:453: RuntimeWarning: Mean of empty slice
  lat=np.nanmean(data_in['latitude'].data,axis=-1)
/home/jovyan/tmp2/ocng-489-689-2026-spring/swot_ssh_utils.py:453: RuntimeWarning: Mean of empty slice
  lat=np.nanmean(data_in['latitude'].data,axis=-1)


  Saved: ./crop_output/cycle_474_polygon_subset_right_rotated.png

Processing cycle 475: SWOT_L2_LR_SSH_Unsmoothed_475_024_20230330T172752_20230330T181858_PGD0_01.nc
Load unsmoothed data to self.left and self.right
Subset data between latitude bounds  [-75, -60]


...

  Saved: ./crop_output/cycle_546_polygon_subset_right_rotated.png

Processing cycle 547: SWOT_L2_LR_SSH_Unsmoothed_547_024_20230610T061318_20230610T070423_PGD0_01.nc
Load unsmoothed data to self.left and self.right
Subset data between latitude bounds  [-75, -60]
  Saved: ./crop_output/cycle_547_polygon_subset_right_rotated.png

Processing cycle 548: SWOT_L2_LR_SSH_Unsmoothed_548_024_20230611T060356_20230611T065501_PGD0_01.nc
Load unsmoothed data to self.left and self.right
Subset data between latitude bounds  [-75, -60]
  Saved: ./crop_output/cycle_548_polygon_subset_right_rotated.png

Processing cycle 549: SWOT_L2_LR_SSH_Unsmoothed_549_024_20230612T055433_20230612T064539_PGD0_01.nc
Load unsmoothed data to self.left and self.right
Subset data between latitude bounds  [-75, -60]
  Saved: ./crop_output/cycle_549_polygon_subset_right_rotated.png

Processing cycle 550: SWOT_L2_LR_SSH_Unsmoothed_550_024_20230613T054511_20230613T063617_PGD0_01.nc
Load unsmoothed data to self.left and self.ri

/home/jovyan/tmp2/ocng-489-689-2026-spring/swot_ssh_utils.py:453: RuntimeWarning: Mean of empty slice
  lat=np.nanmean(data_in['latitude'].data,axis=-1)
/home/jovyan/tmp2/ocng-489-689-2026-spring/swot_ssh_utils.py:453: RuntimeWarning: Mean of empty slice
  lat=np.nanmean(data_in['latitude'].data,axis=-1)


  Saved: ./crop_output/cycle_569_polygon_subset_right_rotated.png

Processing cycle 570: SWOT_L2_LR_SSH_Unsmoothed_570_024_20230703T023749_20230703T032855_PGD0_01.nc
Load unsmoothed data to self.left and self.right
Subset data between latitude bounds  [-75, -60]


/home/jovyan/tmp2/ocng-489-689-2026-spring/swot_ssh_utils.py:453: RuntimeWarning: Mean of empty slice
  lat=np.nanmean(data_in['latitude'].data,axis=-1)
/home/jovyan/tmp2/ocng-489-689-2026-spring/swot_ssh_utils.py:453: RuntimeWarning: Mean of empty slice
  lat=np.nanmean(data_in['latitude'].data,axis=-1)


  Saved: ./crop_output/cycle_570_polygon_subset_right_rotated.png

Processing cycle 571: SWOT_L2_LR_SSH_Unsmoothed_571_024_20230704T022827_20230704T031933_PGD0_01.nc
Load unsmoothed data to self.left and self.right
Subset data between latitude bounds  [-75, -60]
  Saved: ./crop_output/cycle_571_polygon_subset_right_rotated.png

Processing cycle 572: SWOT_L2_LR_SSH_Unsmoothed_572_024_20230705T021905_20230705T031011_PGD0_01.nc
Load unsmoothed data to self.left and self.right
Subset data between latitude bounds  [-75, -60]
  Saved: ./crop_output/cycle_572_polygon_subset_right_rotated.png

Processing cycle 573: SWOT_L2_LR_SSH_Unsmoothed_573_024_20230706T020943_20230706T030049_PGD0_01.nc
Load unsmoothed data to self.left and self.right
Subset data between latitude bounds  [-75, -60]
  Saved: ./crop_output/cycle_573_polygon_subset_right_rotated.png

Processing cycle 574: SWOT_L2_LR_SSH_Unsmoothed_574_024_20230707T020021_20230707T025127_PGD0_01.nc
Load unsmoothed data to self.left and self.ri

In [17]:
'''
Plot a timeline figure with rotated, tightly-cropped SWOT SSH images placed along
the y-axis according to their acquisition date.
'''

from datetime import datetime
import matplotlib.dates as mdates
from matplotlib.offsetbox import OffsetImage, AnnotationBbox
from PIL import Image

data_dir   = "./swot_data"
image_dir  = "./crop_output"
side       = "right"

cycle_min  = 474
cycle_max  = 577

# Reference ("day 0") — day counter starts here.
# Set to None to use the earliest cycle in the range as day 0.
reference_date = None

# Visual options
image_zoom       = 0.35
figsize          = (20, 30)   # wider figure so labels have room
x_fixed          = 0.50       # x-position of image centers
date_label_fmt   = "%Y-%m-%d"

# Label controls
show_label_every = 2          # show every 2nd label; set to 1 for every image
label_fontsize   = 6
label_x_offset   = 150        # points to the right of image center
label_bbox_alpha = 0.75

# White-trim sensitivity: pixels with all RGB channels above this value are
# treated as "white background" and removed.
white_threshold  = 240

output_path      = f"./crop_output/timeline_cycles_{cycle_min}_{cycle_max}_{side}.png"

# ============================================================
# Filename parsing
# ============================================================
NC_PATTERN = re.compile(
    r"SWOT_L2_LR_SSH_Unsmoothed_(\d+)_\d+_(\d{8}T\d{6})_",
    re.IGNORECASE,
)

def parse_nc_filename(fn):
    m = NC_PATTERN.search(os.path.basename(fn))
    if not m:
        return None, None
    cycle = int(m.group(1))
    dt = datetime.strptime(m.group(2), "%Y%m%dT%H%M%S")
    return cycle, dt

def build_cycle_to_date(data_dir, cycle_min, cycle_max):
    cycle_to_dt = {}
    for fn in os.listdir(data_dir):
        if not fn.lower().endswith(".nc"):
            continue
        cycle, dt = parse_nc_filename(fn)
        if cycle is None:
            continue
        if cycle_min <= cycle <= cycle_max:
            if cycle not in cycle_to_dt or dt < cycle_to_dt[cycle]:
                cycle_to_dt[cycle] = dt
    return cycle_to_dt

def find_image_for_cycle(image_dir, cycle, side):
    candidates = [
        f"cycle_{cycle:03d}_polygon_subset_{side}_rotated.png",
        f"cycle_{cycle}_polygon_subset_{side}_rotated.png",
    ]
    for name in candidates:
        path = os.path.join(image_dir, name)
        if os.path.exists(path):
            return path
    return None

# ============================================================
# White padding removal + trim
# ============================================================
def make_white_transparent_and_trim(pil_img, white_thresh=240):
    """
    1. Convert to RGBA.
    2. Set alpha=0 for any pixel whose R,G,B are all >= white_thresh.
    3. Crop to the bounding box of non-transparent content.
    """
    img = pil_img.convert("RGBA")
    arr = np.array(img)

    r, g, b, a = arr[..., 0], arr[..., 1], arr[..., 2], arr[..., 3]
    white_mask = (r >= white_thresh) & (g >= white_thresh) & (b >= white_thresh)
    arr[..., 3] = np.where(white_mask, 0, a).astype(np.uint8)

    img_t = Image.fromarray(arr, mode="RGBA")
    bbox = img_t.getbbox()
    if bbox is not None:
        img_t = img_t.crop(bbox)

    return img_t

# ============================================================
# Gather data
# ============================================================
cycle_to_dt = build_cycle_to_date(data_dir, cycle_min, cycle_max)
if not cycle_to_dt:
    raise RuntimeError(
        f"No NC files found in {data_dir} for cycles [{cycle_min}, {cycle_max}]"
    )

records = []
missing_images = []
for cycle in sorted(cycle_to_dt.keys()):
    dt = cycle_to_dt[cycle]
    img_path = find_image_for_cycle(image_dir, cycle, side)
    if img_path is None:
        missing_images.append(cycle)
        continue
    records.append({"cycle": cycle, "datetime": dt, "image": img_path})

if not records:
    raise RuntimeError("No cycles with both an NC file and a rotated image were found.")

print(f"Matched {len(records)} cycles with images.")
if missing_images:
    print(f"Warning: {len(missing_images)} cycles had no rotated image: {missing_images}")

if reference_date is None:
    reference_date = min(r["datetime"] for r in records)
    reference_date = datetime(reference_date.year, reference_date.month, reference_date.day)

for r in records:
    r["day"] = (r["datetime"] - reference_date).days

# ============================================================
# Plot
# ============================================================
fig, ax = plt.subplots(figsize=figsize)

dates = [r["datetime"] for r in records]
xs = [x_fixed] * len(records)

# central timeline
ax.plot(xs, dates, color="lightgray", linewidth=0.8, zorder=1)

for i, r in enumerate(records):
    raw = Image.open(r["image"])
    img = make_white_transparent_and_trim(raw, white_thresh=white_threshold)

    im = OffsetImage(img, zoom=image_zoom)
    ab = AnnotationBbox(
        im,
        (x_fixed, mdates.date2num(r["datetime"])),
        frameon=False,
        xycoords=("data", "data"),
        box_alignment=(0.5, 0.5),
        pad=0.0,
        zorder=2,
    )
    ax.add_artist(ab)

    # Show only every Nth caption to reduce overlap.
    # Always show first and last too.
    show_label = (i % show_label_every == 0) or (i == 0) or (i == len(records) - 1)

    if show_label:
        label = (
            f"{r['datetime'].strftime(date_label_fmt)}  "
            f"(day {r['day']}, cyc {r['cycle']})"
        )
        ax.annotate(
            label,
            xy=(x_fixed, mdates.date2num(r["datetime"])),
            xytext=(label_x_offset, 0),
            textcoords="offset points",
            va="center",
            ha="left",
            fontsize=label_fontsize,
            bbox=dict(
                facecolor="white",
                edgecolor="none",
                alpha=label_bbox_alpha,
                pad=0.4,
            ),
            arrowprops=dict(
                arrowstyle="-",
                color="gray",
                lw=0.5,
                alpha=0.5,
                shrinkA=0,
                shrinkB=5,
            ),
            zorder=10,
        )

ax.yaxis_date()
ax.yaxis.set_major_locator(mdates.AutoDateLocator())
ax.yaxis.set_major_formatter(mdates.DateFormatter(date_label_fmt))

y_min = mdates.date2num(min(dates))
y_max = mdates.date2num(max(dates))
pad = max(1.0, (y_max - y_min) * 0.02)
ax.set_ylim(y_min - pad, y_max + pad)

# Expand horizontal room:
# left side for y-axis, center for images, right side for labels
ax.set_xlim(0, 1.45)
ax.set_xticks([])

for spine in ("top", "right", "bottom"):
    ax.spines[spine].set_visible(False)

ax.set_ylabel(f"Time   (day 0 = {reference_date.strftime(date_label_fmt)})")
ax.set_title(
    f"SWOT SSHA cropped subsets — cycles {cycle_min}–{cycle_max} ({side} swath)"
)

ax.invert_yaxis()

plt.tight_layout()
os.makedirs(os.path.dirname(output_path), exist_ok=True)
fig.savefig(output_path, dpi=200, bbox_inches="tight")
plt.close(fig)

print(f"Saved timeline figure: {output_path}")

Matched 98 cycles with images.
Saved timeline figure: ./crop_output/timeline_cycles_474_577_right.png
